In [1]:
import os
import numpy as np
import pandas as pd
import scipy.signal as signal
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...

In [3]:
if not os.path.exists(CACHE_FILE):
    raise FileNotFoundError(f"Cache file '{CACHE_FILE}' not found. Please run pre_processing.ipynb first!")

print(f"Loading cached dataset from {CACHE_FILE}...")
cached = np.load(CACHE_FILE, allow_pickle=True)

X_train = cached["X_train"]          
y_train = cached["y_train"]          
X_test = cached["X_test"]            
y_test = cached["y_test"]            
class_names = cached["class_names"]  

# Check if tabular features are in the cache
if "X_feat_train" in cached.files:
    print("Loading tabular features from cache...")
    X_feat_train = cached["X_feat_train"]
    X_feat_test = cached["X_feat_test"]
else:
    print("Tabular features missing from cache. Computing them now (this will take a moment)...")
    
    def extract_ecg_features(signal_3lead, fs=100):
        lead1 = signal_3lead[:, 0]
        lead2 = signal_3lead[:, 1]
        lead3 = signal_3lead[:, 2]

        b, a = signal.butter(2, [5 / (fs / 2), 15 / (fs / 2)], btype='bandpass')
        filtered = signal.filtfilt(b, a, lead1)
        integrated = np.convolve(np.diff(filtered)**2, np.ones(int(0.08 * fs)), mode='same')
        peaks, _ = signal.find_peaks(integrated, distance=int(0.35 * fs), prominence=np.std(integrated) * 0.5)

        rr_intervals = (np.diff(peaks) / fs) * 1000.0 if len(peaks) >= 2 else np.array([800.0])
        hr_bpm = 60000.0 / rr_intervals
        
        mean_hr, min_hr, max_hr, hr_std = np.mean(hr_bpm), np.min(hr_bpm), np.max(hr_bpm), np.std(hr_bpm)
        mean_rr, rr_std = np.mean(rr_intervals), np.std(rr_intervals)
        rr_range = np.max(rr_intervals) - np.min(rr_intervals)
        rr_cv = (rr_std / mean_rr) if mean_rr > 0 else 0.0

        sdnn = np.std(rr_intervals)
        rr_diff = np.diff(rr_intervals)
        rmssd = np.sqrt(np.mean(rr_diff ** 2)) if len(rr_diff) > 0 else 0.0
        pnn50 = (np.sum(np.abs(rr_diff) > 50) / len(rr_diff)) * 100.0 if len(rr_diff) > 0 else 0.0

        sample_entropy = 0.5  # Simplified placeholder for speed in fallback
        mean_pr, mean_qrs, mean_qt = 160.0, 90.0, 380.0
        mean_qtc = mean_qt / np.sqrt(mean_rr / 1000.0) if mean_rr > 0 else 400.0
        
        freqs, psd = signal.welch(lead1, fs=fs, nperseg=256)
        bw_power = np.sum(psd[freqs < 0.5])
        total_power = np.sum(psd) + 1e-8
        snr = 10 * np.log10((total_power - bw_power) / (bw_power + 1e-8))
        baseline_wander_mag = bw_power / total_power

        lead_stats = []
        for l in [lead1, lead2, lead3]:
            lead_stats.extend([np.mean(l), np.std(l), skew(l), kurtosis(l), np.ptp(l), np.percentile(l, 75) - np.percentile(l, 25)])

        return np.array([mean_hr, min_hr, max_hr, hr_std, mean_rr, rr_std, rr_range, rr_cv,
                         sdnn, rmssd, pnn50, sample_entropy, mean_pr, mean_qrs, mean_qt, mean_qtc,
                         snr, baseline_wander_mag] + lead_stats, dtype=np.float32)

    X_feat_train = np.nan_to_num(np.array([extract_ecg_features(x) for x in X_train]))
    X_feat_test = np.nan_to_num(np.array([extract_ecg_features(x) for x in X_test]))
    
    print("Updating cache with computed features...")
    np.savez_compressed(
        CACHE_FILE,
        X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
        class_names=class_names, X_feat_train=X_feat_train, X_feat_test=X_feat_test
    )

print(f"Loaded successfully!")
print(f"Signals   -> Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features  -> Train: {X_feat_train.shape} | Test: {X_feat_test.shape}")
print(f"Targets   -> Classes ({len(class_names)}): {class_names.tolist()}")

Loading cached dataset from ptbxl_preprocessed_cache.npz...
Tabular features missing from cache. Computing them now (this will take a moment)...
Updating cache with computed features...
Loaded successfully!
Signals   -> Train: (19230, 1000, 3) | Test: (2158, 1000, 3)
Features  -> Train: (19230, 36) | Test: (2158, 36)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']
